In [0]:
#Break from this cell to new Silver layer notebook.
# Reading parquet & create dataframe. 


from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import os

BASE_DIR = '/Volumes/debora_ryan_susheela_hhs/default/upload_volume'
BRONZE_PARQUET_DIR = os.path.join(BASE_DIR, "bronze_output", "parquet_data_hhs")
filepath_rates = os.path.join(BRONZE_PARQUET_DIR, 'rates')

def read_parquet(filepath: str) -> DataFrame:
    data_f = spark.read.parquet(filepath)
    return data_f
    
df_rates = read_parquet(filepath_rates)


In [0]:
from pyspark.sql import functions as F

df_rates_clean_age = df_rates.filter(F.col("Age") != "Family Option").withColumn(
    "Age_int",
    F.when(F.col("Age") == "0-20", F.lit(20))
     .when(F.col("Age") == "65 and over", F.lit(65))
     .otherwise(F.col("Age").cast("int"))
)

display(df_rates_clean_age.select("Age", "Age_int").distinct().orderBy("Age_int"))

In [0]:
display(
    df_rates.groupBy("PlanId")
        .agg(F.count("*").alias("row_count"))
        .orderBy(F.col("row_count").desc())
        .limit(10)
)

In [0]:
df_rates_clean_age_non_smokers = df_rates_clean_age.filter(
    ((F.col("Tobacco") == "No") | (F.col("Tobacco") == "No Preference"))
)

In [0]:
# Write data to parquet
def write(input_df: DataFrame, out_dir):
    return input_df.write.mode('overwrite').parquet(out_dir)

In [0]:
SILVER_PARQUET_DIR = os.path.join(BASE_DIR, "silver", "data", "age")
os.makedirs(SILVER_PARQUET_DIR, exist_ok=True)

write(df_rates_clean_age_non_smokers, f"{SILVER_PARQUET_DIR}/non_smokers")